# Week 12: Transformers — Self-Attention, and Building One in PyTorch

**Topics:** scaled dot-product self-attention **from scratch** (matching the
lecture formula) -> the same computation via `nn.MultiheadAttention` ->
stacking encoder blocks with `nn.TransformerEncoder` and **positional
encoding** -> **train a tiny Transformer** to reverse a sequence and *watch
its attention* -> a 3-line taste of pretrained Transformers with Hugging Face.

Everything this week lives in the **language / sequence** world (no images
yet — ViT comes later). Keep an eye on the **shape** printed in almost every
cell: being able to picture the shape of `X`, `Q`, `scores`, and `attn` at
each step is the whole game.

In [ ]:
import math

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline
torch.manual_seed(0)

### Environment setup

Run this once. On Colab it installs `transformers` (used only in the last
section). PyTorch is already available on Colab.

In [ ]:
import sys, subprocess

IN_COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, "__IPYTHON__") else False

if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])

print(f"Running on: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"PyTorch version: {torch.__version__}")

### Display helpers

`show_attention` draws one attention matrix (query rows x key columns);
`show_map` draws one generic 2-D array; `show_curve` plots one training curve.
As always, each helper shows exactly one figure — call it again for another.

In [ ]:
# Each helper shows exactly one figure.

def show_attention(attn, title=None, x_tokens=None, y_tokens=None, scale=4.5):
    # attn: a 2-D array, rows = query positions, cols = key positions
    n_q, n_k = attn.shape
    fig, ax = plt.subplots(figsize=(scale, scale))
    im = ax.imshow(attn, cmap="viridis", vmin=0.0)
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_xlabel("key position")
    ax.set_ylabel("query position")
    ax.set_xticks(range(n_k))            # integer ticks: 0, 1, 2, ...
    ax.set_yticks(range(n_q))
    if x_tokens is not None:
        ax.set_xticklabels(x_tokens)
    if y_tokens is not None:
        ax.set_yticklabels(y_tokens)
    if title:
        ax.set_title(title)
    plt.tight_layout()
    plt.show()


def show_map(m, title=None, scale=4, cmap="viridis"):
    # m: any 2-D array (positional encoding, feature map, ...)
    fig, ax = plt.subplots(figsize=(scale, scale))
    im = ax.imshow(m, cmap=cmap)
    plt.colorbar(im, ax=ax, fraction=0.046)
    if title:
        ax.set_title(title)
    plt.tight_layout()
    plt.show()


def show_curve(values, title=None, xlabel="step", ylabel="loss", scale=(6, 3)):
    # values: a 1-D sequence to plot (e.g., loss over steps)
    fig, ax = plt.subplots(figsize=scale)
    ax.plot(values)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    plt.tight_layout()
    plt.show()

---

## 1. Self-Attention from Scratch

We rebuild the exact formula from the lecture, one line at a time:

$$Q = X W_Q,\quad K = X W_K,\quad V = X W_V,\qquad \text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Watch the shape at every step.

In [ ]:
# A toy "sentence": 4 tokens, each represented by a 6-dim embedding.
seq_len = 4
d_model = 6

X = torch.randn(seq_len, d_model)
print("X (token embeddings) shape:", X.shape)   # (seq_len, d_model) = (4, 6)
print(f"X =\n{X}")

In [ ]:
# Three independent learned projections. Here d_k = d_model for simplicity.
d_k = 6
W_Q = nn.Linear(d_model, d_k, bias=False)
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_k, bias=False)

Q = W_Q(X)
K = W_K(X)
V = W_V(X)
print("Q shape:", Q.shape)   # (4, 6)
print("K shape:", K.shape)   # (4, 6)
print("V shape:", V.shape)   # (4, 6)

In [ ]:
# Every query dotted with every key -> a (seq_len x seq_len) score matrix.
scores = torch.matmul(Q, K.T) / math.sqrt(d_k)
print("scores shape:", scores.shape)   # (4, 4): row i = query i compared to all keys
print(f"scores =\n{scores}")

In [ ]:
# Softmax over the key axis -> each query's weights sum to 1.
attn = F.softmax(scores, dim=-1)
print("attn shape:", attn.shape)               # (4, 4)
print("row sums (should all be 1):", attn.sum(dim=-1))

In [ ]:
# Weighted sum of the value vectors -> one output vector per query.
out = torch.matmul(attn, V)
print("out shape:", out.shape)   # (4, 6) - same shape as X

In [ ]:
# The attention weights are random here (untrained), so there is no structure.
show_attention(attn.detach().numpy(), title="Self-attention weights (untrained)")

That is the entire mechanism: **three projections, one matmul, a softmax,
one more matmul.** The weights are random here, so the map is meaningless —
structure appears only after *training* (Section 4).

---

## 2. The Same Thing: `nn.MultiheadAttention`

PyTorch already packages Section 1 (plus an output projection) into one
module. The only new things to learn are its **input/output shapes** and a
couple of arguments.

In [ ]:
# embed_dim must match d_model; batch_first=True -> shapes are (batch, seq, dim).
mha = nn.MultiheadAttention(embed_dim=6, num_heads=1, batch_first=True)

# nn.MultiheadAttention expects a batch dimension, so add one.
X_batched = X.unsqueeze(0)
print("X_batched shape:", X_batched.shape)   # (1, 4, 6) = (batch, seq, dim)

In [ ]:
# Self-attention: query, key, and value are all the same sequence.
attn_out, attn_weights = mha(X_batched, X_batched, X_batched)
print("attn_out shape:    ", attn_out.shape)      # (1, 4, 6) - one vector per token
print("attn_weights shape:", attn_weights.shape)  # (1, 4, 4) - the attention matrix

The structure is identical to Section 1 — only the *values* differ, because
`mha` has its own random `W_Q, W_K, W_V` inside.

In [ ]:
# attn_weights[0] is the same (seq x seq) matrix we built by hand in Section 1.
show_attention(attn_weights[0].detach().numpy(), title="nn.MultiheadAttention weights")

In [ ]:
# More heads = several attention maps in parallel. embed_dim must be divisible by num_heads.
mha_multi = nn.MultiheadAttention(embed_dim=8, num_heads=4, batch_first=True)

X8 = torch.randn(1, 4, 8)
print("X8 shape:", X8.shape)   # (1, 4, 8)

# average_attn_weights=False keeps each head's map separate.
attn_out, attn_weights = mha_multi(X8, X8, X8, average_attn_weights=False)
print("attn_out shape:    ", attn_out.shape)      # (1, 4, 8)
print("attn_weights shape:", attn_weights.shape)  # (1, 4, 4, 4) = (batch, heads, seq, seq)

In [ ]:
# Each head is its own attention map (here untrained, so just different random patterns).
show_attention(attn_weights[0, 0].detach().numpy(), title="Head 0")
show_attention(attn_weights[0, 1].detach().numpy(), title="Head 1")

---

## 3. Stacking Blocks + Positional Encoding

`nn.TransformerEncoderLayer` bundles multi-head self-attention with a
feed-forward network and residual + LayerNorm. `nn.TransformerEncoder` stacks
several of them. This is how you **declare a Transformer encoder of any
size** — just choose `d_model`, `nhead`, `dim_feedforward`, `num_layers`.

In [ ]:
# Declare a 2-layer encoder. dropout=0.0 keeps this demo deterministic.
layer = nn.TransformerEncoderLayer(
    d_model=16, nhead=4, dim_feedforward=64, batch_first=True, dropout=0.0
)
encoder = nn.TransformerEncoder(layer, num_layers=2)
encoder.eval()

x = torch.randn(1, 5, 16)
print("input shape: ", x.shape)    # (1, 5, 16) = (batch, seq, d_model)
y = encoder(x)
print("output shape:", y.shape)    # (1, 5, 16) - sequence in, sequence out (same shape)

In [ ]:
# Self-attention is a SET operation: shuffle the tokens and the outputs just shuffle too.
perm = [2, 0, 4, 1, 3]
y_perm = encoder(x[:, perm, :])
print("encoder(shuffled) == shuffle(encoder(x)) ?",
      torch.allclose(y_perm, y[:, perm, :], atol=1e-5))

So the encoder has **no notion of order**. But order matters
("dog bites man" != "man bites dog"). The fix: add a **positional encoding**
to each token before the encoder sees it.

In [ ]:
def positional_encoding(seq_len, d_model):
    position = torch.arange(seq_len).unsqueeze(1)                       # (seq_len, 1)
    div_term = torch.pow(10000, torch.arange(0, d_model, 2) / d_model)  # (d_model/2,)
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position / div_term)   # even dims: sine
    pe[:, 1::2] = torch.cos(position / div_term)   # odd dims:  cosine
    return pe

pe_vis = positional_encoding(seq_len=32, d_model=16)
print("pe_vis shape:", pe_vis.shape)   # (32, 16): one row per position
show_map(pe_vis.numpy(), title="Sinusoidal positional encoding (32 positions x 16 dims)")

In [ ]:
# Add positional encoding to the tokens, THEN run the encoder.
pe5 = positional_encoding(seq_len=5, d_model=16)
x_pe = x + pe5
y_pe = encoder(x_pe)
y_pe_perm = encoder(x[:, perm, :] + pe5)

# Now shuffling the tokens does NOT simply shuffle the output: position matters.
print("still permutation-invariant ?",
      torch.allclose(y_pe_perm, y_pe[:, perm, :], atol=1e-5))

---

## 4. Train a Tiny Transformer to *Reverse* a Sequence

Now we put it together and actually train. Task: given a sequence of digits,
output it **reversed** (`[3, 1, 4, 1, 5, 9] -> [9, 5, 1, 4, 1, 3]`). No text
dataset and no tokenizer — we generate random digit sequences on the fly.

Model: **embedding + positional encoding -> one self-attention layer -> a
linear readout** that predicts a digit at each position. We keep a *single*
attention layer so that the attention map *is* the model's reasoning — and we
can read it directly.

In [ ]:
VOCAB_SIZE = 10   # digits 0-9
SEQ_LEN = 6

def make_batch(batch_size):
    x = torch.randint(0, VOCAB_SIZE, (batch_size, SEQ_LEN))
    y = torch.flip(x, dims=[1])   # the reversed target
    return x, y

x, y = make_batch(batch_size=2)
print("x shape:", x.shape)   # (2, 6)
print("x[0] =", x[0].tolist())
print("y[0] =", y[0].tolist(), " <- reversed")

In [ ]:
class ReverseTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, seq_len):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos", positional_encoding(seq_len, d_model))  # fixed, not learned
        self.attn = nn.MultiheadAttention(d_model, num_heads=1, batch_first=True)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        h = self.embed(x)            # (B, L, d_model)
        h = h + self.pos             # add positional encoding
        h, _ = self.attn(h, h, h)    # self-attention: (B, L, d_model)
        logits = self.head(h)        # (B, L, vocab_size)
        return logits

In [ ]:
model = ReverseTransformer(vocab_size=VOCAB_SIZE, d_model=64, seq_len=SEQ_LEN)

# One untrained forward pass to verify the shapes line up.
x, y = make_batch(batch_size=2)
logits = model(x)
print("logits shape:", logits.shape)   # (2, 6, 10) = (batch, seq, vocab)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
criterion = nn.CrossEntropyLoss()

losses = []
for step in range(1500):
    x, y = make_batch(batch_size=64)
    logits = model(x)                                   # (B, L, V)

    # CrossEntropyLoss wants (N, V) and (N,), so flatten the batch and sequence axes.
    loss = criterion(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    if step % 300 == 0:
        print(f"step {step:4d} | loss {loss.item():.4f}")

show_curve(losses, title="Training loss", ylabel="loss")

In [ ]:
model.eval()
x, y = make_batch(batch_size=1000)
logits = model(x)
pred = logits.argmax(dim=-1)               # (1000, 6)
accuracy = (pred == y).float().mean()
print("token-level accuracy:", accuracy.item())

print("input:    ", x[0].tolist())
print("target:   ", y[0].tolist())
print("predicted:", pred[0].tolist())

In [ ]:
# Re-run the attention layer with need_weights=True to grab the learned attention map.
model.eval()
x, y = make_batch(batch_size=1)
h = model.embed(x) + model.pos
_, attn_weights = model.attn(h, h, h, need_weights=True)
print("attn_weights shape:", attn_weights.shape)   # (1, 6, 6)

tokens = x[0].tolist()
show_attention(
    attn_weights[0].detach().numpy(),
    title="Learned attention (reverse task)",
    x_tokens=tokens,
    y_tokens=list(reversed(tokens)),
)

The bright cells form an **anti-diagonal**: output position 0 attends to the
*last* input position, position 1 to the second-last, and so on. The model
discovered "to reverse, copy from the mirrored position" — and the attention
map shows it doing exactly that.

### Exercise 4.1 — A different task, a different map

Train the **same** model on a different sequence-to-sequence task, and predict
what its attention map will look like *before* you plot it.

Pick one:
- **copy**: target = input, unchanged
- **shift**: target = input rolled by one position (`torch.roll(x, shifts=1, dims=1)`)

Write a `make_batch`-style function with your new target, create a fresh
`model`, retrain it (copy the training loop), and visualize the attention with
the same code as above.

**Predict first:** where will the bright cells land for your task?

In [ ]:
# YOUR CODE HERE
# 1. Write make_batch_task(batch_size) returning (x, y) for your chosen task.
# 2. Create a fresh model = ReverseTransformer(...), then retrain it (copy the loop above).
# 3. Visualize the learned attention with show_attention.


# Where did the bright cells land, and why does that pattern solve your task?
# (You may write your answer in Korean.)
#

### Exercise 4.2 — Declare your own encoder

Rebuild the model using `nn.TransformerEncoder` (Section 3) in place of the
single `nn.MultiheadAttention` — your choice of `nhead`, `num_layers`,
`dim_feedforward`. Add positional encoding before the encoder and keep the
`nn.Linear` readout head. Train it on the reverse task and report the
token-level accuracy.

This is the "declare any Transformer you want" tool — the same `forward /
backward / step` loop trains it.

In [ ]:
# YOUR CODE HERE
# Build a model class that uses nn.TransformerEncoder, train it on the reverse
# task, and print the token-level accuracy.

---

## 5. Bonus: Pretrained Transformers in 3 Lines (Hugging Face)

You built and trained a Transformer from its parts. In practice you rarely
start from scratch — you download a model someone already trained on huge
amounts of data. The **Hugging Face `transformers`** library makes this a
one-liner. (Next week's CLIP is the same idea, for images + text.)

In [ ]:
from transformers import pipeline

# Downloads a small pretrained Transformer the first time it runs.
classifier = pipeline("sentiment-analysis")

print(classifier("This lab finally made attention click for me!"))
print(classifier("I was so confused by the math at first."))

---

### Wrap-up

- Self-attention = **three projections, a scaled dot-product, a softmax, a weighted sum** — you wrote it by hand and watched `nn.MultiheadAttention` do the same.
- `nn.TransformerEncoder` + **positional encoding** let you declare and train a Transformer of any size with the familiar `forward / backward / step` loop.
- A trained attention map is **readable**: the reverse task produced a clean anti-diagonal.
- Pretrained Transformers are one `pipeline(...)` away — the bridge to CLIP next week.